# Tutorial

This tutorial walks through the core workflow of `molisanax`:

1. Build a synthetic forcing made of an ocean surface jet and an overlying wind, wrap it in a `Dataset`.
2. Run a **deterministic** trajectory through the field with the `solve` ODE integrator, including a windage term.
3. Run a **stochastic** ensemble using a Smagorinsky-style diffusion — exercising the SDE mode of `solve` and the `Dataset.neighborhood` API.
4. **Learn** the windage coefficient by JAX automatic differentiation, using `optimistix` and the separation distance as a residual.
5. **Jointly learn** the windage and the Smagorinsky coefficient on a stochastic simulator, using a time-aggregated energy score with separation distance as kernel.

The same pattern transposes to real forcing fields loaded from xarray (zarr / netCDF) via `Dataset.from_xarray`.

> Boilerplate cells (imports, plot setup, animation rendering) are folded by default — click *Show code* to expand. The cells most relevant to the `molisanax` API are always expanded.

In [ ]:
import jax
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import jax.random as jr
import equinox as eqx
import numpy as np
import cmocean
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML


## 1. Defining the forcing fields

We build two synthetic fields on the same grid:

- a meandering zonal **ocean jet** — a Bickley-style jet whose centerline
  $y_c(x, t)$ slowly meanders, giving a realistic-looking surface current $\mathbf{u}_o$;
- a slowly **rotating wind** $\mathbf{u}_w$ — spatially uniform, time-varying, deliberately
  decorrelated from the ocean current.

Both fields share the same grid (101 × 101 × 97), spanning two degrees in lat/lon and four days in time.

In [ ]:
from molisanax import degrees_to_meters

ny = nx = 101
nt = 97  # 4 days, hourly

lat = jnp.linspace(29.0, 31.0, ny)
lon = jnp.linspace(-1.0, 1.0, nx)

dt = 60 * 60                                # one hour, in seconds
ts = jnp.linspace(0.0, dt * (nt - 1), nt)   # 0 .. 96 h

dy_m, dx_m = degrees_to_meters(
    jnp.asarray([lat[1] - lat[0], lon[1] - lon[0]]), lat.mean()
)


In [ ]:
# Meandering Bickley jet -- u_o, v_o in m/s on the (nt, ny, nx) grid
U0      = 0.5                              # peak jet speed (m/s)
lat_c   = 30.0                             # mean jet latitude
L_jet   = 0.15                             # jet half-width (deg)
A_meand = 0.10                             # meander amplitude (deg lat)
k_meand = 2.0 * jnp.pi / 1.0               # meander wavenumber (1/deg lon)
T_meand = 4.0 * 86400.0                    # meander period (s)

LON, LAT = jnp.meshgrid(lon, lat, indexing="xy")  # both (ny, nx)
deg_ratio = (dy_m / (lat[1] - lat[0])) / (dx_m / (lon[1] - lon[0]))  # m/deg ratio

def jet_uv(t):
    phase = k_meand * LON - 2 * jnp.pi * t / T_meand
    yc = lat_c + A_meand * jnp.sin(phase)
    sech2 = 1.0 / jnp.cosh((LAT - yc) / L_jet) ** 2
    # tangent of meander centerline in m/m (dimensionless slope)
    dyc_dx = A_meand * k_meand * jnp.cos(phase) * deg_ratio
    return U0 * sech2, U0 * sech2 * dyc_dx     # (u_o, v_o) in m/s

u_o, v_o = jax.vmap(jet_uv)(ts)
print("u_o, v_o shape:", u_o.shape,
      " peak speed:", float(jnp.sqrt(u_o ** 2 + v_o ** 2).max()), "m/s")


In [ ]:
# Slowly-rotating uniform wind, decorrelated from the ocean current
W       = 8.0                               # wind speed magnitude (m/s)
T_wind  = 4.0 * 86400.0                     # full rotation in 4 days
alpha0  = jnp.pi / 4                        # initial direction (NE)

def wind_uv(t):
    alpha = alpha0 + 2 * jnp.pi * t / T_wind
    return (W * jnp.cos(alpha) * jnp.ones_like(LAT),
            W * jnp.sin(alpha) * jnp.ones_like(LAT))

u_w, v_w = jax.vmap(wind_uv)(ts)
print("u_w, v_w shape:", u_w.shape, " |u_w|:", float(W), "m/s (uniform)")


Animation of the ocean jet — speed in colour, arrows show direction.

In [ ]:
ocean_speed = jnp.sqrt(u_o ** 2 + v_o ** 2)
ocean_clim = float(ocean_speed.max())

fig, ax = plt.subplots(figsize=(5, 5))
plt.close(fig)

im = ax.pcolormesh(lon, lat, ocean_speed[0], cmap=cmocean.cm.speed,
                   vmin=0, vmax=ocean_clim)
quiv = ax.quiver(lon[::6], lat[::6], u_o[0][::6, ::6], v_o[0][::6, ::6],
                 scale=20, color="white", alpha=0.7)
fig.colorbar(im, ax=ax, label=r"$\| \mathbf{u}_o \|$  (m s$^{-1}$)")
ax.set_aspect("equal")
ax.set_xlabel("lon"); ax.set_ylabel("lat")
title = ax.set_title(f"$t$ = {ts[0] / 3600:.0f} h")

def draw(k):
    im.set_array(np.asarray(ocean_speed[k]).ravel())
    quiv.set_UVC(np.asarray(u_o[k][::6, ::6]), np.asarray(v_o[k][::6, ::6]))
    title.set_text(f"$t$ = {ts[k] / 3600:.0f} h")
    return im, quiv, title

HTML(animation.FuncAnimation(fig, draw, frames=nt, interval=80, blit=False).to_jshtml())


Animation of the wind field — magnitude is uniform, direction rotates over the 4-day window.

In [ ]:
wind_speed = jnp.sqrt(u_w ** 2 + v_w ** 2)

fig, ax = plt.subplots(figsize=(5, 5))
plt.close(fig)

im = ax.pcolormesh(lon, lat, wind_speed[0], cmap=cmocean.cm.amp,
                   vmin=0, vmax=float(wind_speed.max()))
quiv = ax.quiver(lon[::8], lat[::8], u_w[0][::8, ::8], v_w[0][::8, ::8],
                 scale=120, color="black", alpha=0.7)
fig.colorbar(im, ax=ax, label=r"$\| \mathbf{u}_w \|$  (m s$^{-1}$)")
ax.set_aspect("equal")
ax.set_xlabel("lon"); ax.set_ylabel("lat")
title = ax.set_title(f"$t$ = {ts[0] / 3600:.0f} h")

def draw(k):
    im.set_array(np.asarray(wind_speed[k]).ravel())
    quiv.set_UVC(np.asarray(u_w[k][::8, ::8]), np.asarray(v_w[k][::8, ::8]))
    title.set_text(f"$t$ = {ts[k] / 3600:.0f} h")
    return im, quiv, title

HTML(animation.FuncAnimation(fig, draw, frames=nt, interval=80, blit=False).to_jshtml())


Wrap both fields into a single `Dataset` so they can be queried by the integrator.
`Dataset.from_arrays` accepts plain numpy or JAX arrays directly; for real forcings,
use `Dataset.from_xarray` instead.

In [ ]:
from molisanax import Dataset

forcing = Dataset.from_arrays(
    fields={"u_o": u_o, "v_o": v_o, "u_w": u_w, "v_w": v_w},
    t=ts,
    lat=lat,
    lon=lon,
    dtype=jnp.float64,
)


## 2. Simulating trajectories

### 2.1 A single deterministic trajectory with windage

A surface object is advected by the ocean current and partially dragged by the wind. The
**direct windage** model parameterises this as

$$
\mathrm{d}\mathbf{X}(t) = \bigl[\mathbf{u}_o(t, \mathbf{X}) + \beta_w \, \mathbf{u}_w(t, \mathbf{X})\bigr] \, \mathrm{d}t,
$$

with $\beta_w$ a small dimensionless coefficient — typically $0.1$ to $10\%$ depending on
the object. Here we take $\beta_w = 3\%$ as ground truth.

In `molisanax` the dynamics are expressed as a Python callable `term(t, y, args)` that
returns the velocity in degrees per second. `solve` defaults to ODE mode and uses the
`Heun` integrator with a fixed step size.

In [ ]:
from molisanax import solve, Heun, meters_to_degrees

BETA_W_TRUE = 0.03  # 3% direct windage

def windage_term(t, y, args):
    dataset, beta_w = args
    lat_, lon_ = y[0], y[1]
    uo = dataset["u_o"].interp(t, lat_, lon_)
    vo = dataset["v_o"].interp(t, lat_, lon_)
    uw = dataset["u_w"].interp(t, lat_, lon_)
    vw = dataset["v_w"].interp(t, lat_, lon_)
    u = uo + beta_w * uw
    v = vo + beta_w * vw
    # convert (north=v, east=u) m/s -> (dlat/dt, dlon/dt) deg/s
    return meters_to_degrees(jnp.array([v, u]), lat_)

y0 = jnp.array([29.95, -0.5])           # initial [lat, lon] -- just south of jet axis
ts_sim = ts[1:-1]                       # avoid boundary timestamps

traj = solve(windage_term, (forcing, BETA_W_TRUE), y0, ts_sim, Heun())
print("trajectory shape:", traj.shape)


Animation — the time-evolving ocean speed underneath, the trajectory growing one step at a time.

In [ ]:
traj_np = np.asarray(traj)

fig, ax = plt.subplots(figsize=(5, 5))
plt.close(fig)

im = ax.pcolormesh(lon, lat, ocean_speed[0], cmap=cmocean.cm.speed,
                   vmin=0, vmax=ocean_clim)
fig.colorbar(im, ax=ax, label=r"$\| \mathbf{u}_o \|$  (m s$^{-1}$)")
line, = ax.plot([], [], color="tab:orange", lw=2)
head = ax.scatter([], [], color="white", edgecolor="black", zorder=3)
ax.set_aspect("equal")
ax.set_xlim(float(lon.min()), float(lon.max()))
ax.set_ylim(float(lat.min()), float(lat.max()))
ax.set_xlabel("lon"); ax.set_ylabel("lat")
title = ax.set_title("")

# match field timestep to trajectory timestep (ts_sim starts at index 1)
def draw(k):
    field_k = k + 1
    im.set_array(np.asarray(ocean_speed[field_k]).ravel())
    line.set_data(traj_np[: k + 1, 1], traj_np[: k + 1, 0])
    head.set_offsets(np.asarray([[traj_np[k, 1], traj_np[k, 0]]]))
    title.set_text(f"$t$ = {ts_sim[k] / 3600:.0f} h")
    return im, line, head, title

HTML(animation.FuncAnimation(fig, draw, frames=len(traj_np), interval=80, blit=False).to_jshtml())


### 2.2 A stochastic ensemble

Sub-grid turbulence the gridded $\mathbf{u}_o$ does not resolve is conventionally
reintroduced as a stochastic diffusion with a Smagorinsky-style local turbulent
viscosity:

$$
K(\mathbf{x}, t) = (C_S \, \Delta x)^2 \sqrt{2(\partial_x u)^2 + 2(\partial_y v)^2 + (\partial_y u + \partial_x v)^2},
$$

estimated from a $3 \times 3$ patch of the current via `Dataset.neighborhood`. The
noise amplitude is $\sqrt{2K / \Delta t}$ so that the per-step displacement variance
is $2K \, \Delta t$.

We switch to SDE mode by passing `key` and `n_noise` to `solve`. The term receives a
pre-sampled noise vector $\mathbf{z}$ as a fourth argument and returns the **full
velocity** (drift + noise term).

In [ ]:
from molisanax import safe_sqrt

CS_TRUE = 0.2  # ground-truth Smagorinsky coefficient (will be relearned in 3.2)
dt_step = float(ts_sim[1] - ts_sim[0])

def smag_windage_term(t, y, args, z):
    dataset, beta_w, c_s = args
    lat_, lon_ = y[0], y[1]

    uo = dataset["u_o"].interp(t, lat_, lon_)
    vo = dataset["v_o"].interp(t, lat_, lon_)
    uw = dataset["u_w"].interp(t, lat_, lon_)
    vw = dataset["v_w"].interp(t, lat_, lon_)
    u = uo + beta_w * uw
    v = vo + beta_w * vw
    drift = meters_to_degrees(jnp.array([v, u]), lat_)

    patches = dataset.neighborhood(t, lat_, lon_, t_window=0, lat_window=1, lon_window=1)
    u_p = patches["u_o"][0]
    v_p = patches["v_o"][0]
    du_dx = (u_p[1, 2] - u_p[1, 0]) / (2 * dx_m)
    du_dy = (u_p[2, 1] - u_p[0, 1]) / (2 * dy_m)
    dv_dx = (v_p[1, 2] - v_p[1, 0]) / (2 * dx_m)
    dv_dy = (v_p[2, 1] - v_p[0, 1]) / (2 * dy_m)

    strain = safe_sqrt(2 * du_dx ** 2 + 2 * dv_dy ** 2 + (du_dy + dv_dx) ** 2)
    K = (c_s * dx_m) ** 2 * strain
    sigma = safe_sqrt(2 * K / dt_step)

    return drift + meters_to_degrees(sigma * z, lat_)

ensemble = solve(
    smag_windage_term, (forcing, BETA_W_TRUE, CS_TRUE), y0, ts_sim,
    key=jr.key(0), n_noise=2, n_samples=100,
)
print("ensemble shape:", ensemble.shape)


Animation — the ensemble particle cloud spreading on top of the time-evolving ocean speed.

In [ ]:
ens_np = np.asarray(ensemble)

bin_x = np.linspace(float(lon.min()), float(lon.max()), 41)
bin_y = np.linspace(float(lat.min()), float(lat.max()), 41)
H_max = max(np.histogram2d(ens_np[:, k, 1], ens_np[:, k, 0], bins=[bin_x, bin_y])[0].max()
            for k in range(ens_np.shape[1]))

fig, ax = plt.subplots(figsize=(5, 5))
plt.close(fig)

bg = ax.pcolormesh(lon, lat, ocean_speed[0], cmap=cmocean.cm.speed,
                   vmin=0, vmax=ocean_clim, alpha=0.6)
H0, _, _ = np.histogram2d(ens_np[:, 0, 1], ens_np[:, 0, 0], bins=[bin_x, bin_y])
hist = ax.pcolormesh(bin_x, bin_y, H0.T, cmap=cmocean.cm.amp, vmin=0, vmax=H_max, alpha=0.85)
det_line, = ax.plot([], [], color="tab:orange", lw=1.5)
ax.set_aspect("equal")
ax.set_xlabel("lon"); ax.set_ylabel("lat")
fig.colorbar(hist, ax=ax, label="ensemble count")
title = ax.set_title("")

def draw(k):
    field_k = k + 1
    bg.set_array(np.asarray(ocean_speed[field_k]).ravel())
    H, _, _ = np.histogram2d(ens_np[:, k, 1], ens_np[:, k, 0], bins=[bin_x, bin_y])
    hist.set_array(H.T.ravel())
    det_line.set_data(traj_np[: k + 1, 1], traj_np[: k + 1, 0])
    title.set_text(f"$t$ = {ts_sim[k] / 3600:.0f} h")
    return bg, hist, det_line, title

HTML(animation.FuncAnimation(fig, draw, frames=ens_np.shape[1], interval=80, blit=False).to_jshtml())


## 3. Learning a simulator parameterisation

`molisanax`'s `solve` is fully differentiable. We exploit that to **fit** the parameters
of a term function: given a reference trajectory $\mathbf{Y}^*$ produced by the *true*
dynamics, recover the parameters of a model that match it.

We treat both deterministic and stochastic settings:

- **§3.1 Deterministic.** Recover the windage coefficient $\beta_w$ from a single
  trajectory, using the per-time **separation distance** between simulated and
  reference paths as residuals.
- **§3.2 Stochastic.** Jointly recover $(\beta_w, C_S)$ from an array of observed
  drifter trajectories, using a time-aggregated **energy score** with the separation
  distance as kernel.

### 3.1 Deterministic: learning $\beta_w$

The model uses the (perfectly observed) true ocean and wind fields, but a tunable
$\beta_w$. The reference trajectory is generated with $\beta_w = 0.03$.

In [ ]:
class TunableWindage(eqx.Module):
    beta_w: jax.Array

    def __call__(self, t, y, args):
        dataset = args
        return windage_term(t, y, (dataset, self.beta_w))

term_init = TunableWindage(beta_w=jnp.array(0.0))

# Reference trajectory: true windage on the true ocean+wind dataset
true_traj = solve(windage_term, (forcing, BETA_W_TRUE), y0, ts_sim, Heun(), adjoint="forward")

# Initial estimate: tunable term with beta_w = 0
init_traj = solve(term_init, forcing, y0, ts_sim, Heun(), adjoint="forward")
print("ref shape:", true_traj.shape, "  init shape:", init_traj.shape)


In [ ]:
from molisanax import separation_distance
import optimistix as optx

@eqx.filter_jit
def residual_fn(term_module, ref_traj):
    est_traj = solve(term_module, forcing, y0, ts_sim, Heun(), adjoint="forward")
    return separation_distance(est_traj, ref_traj)  # (T,) residuals in metres

solver = optx.BestSoFarLeastSquares(optx.GaussNewton(rtol=1e-8, atol=1e-8))
sol = optx.least_squares(residual_fn, solver=solver, y0=term_init, args=true_traj)
term_fit = sol.value
print(f"converged in {int(sol.stats['num_steps'])} steps  ->  "
      f"beta_w = {float(term_fit.beta_w) * 100:.3f}%")
print(f"truth                          ->  beta_w = {BETA_W_TRUE * 100:.3f}%")


In [ ]:
final_traj = solve(term_fit, forcing, y0, ts_sim, Heun(), adjoint="forward")

fig, ax = plt.subplots(figsize=(5, 5), constrained_layout=True)
im = ax.pcolormesh(lon, lat, ocean_speed[len(ocean_speed) // 2], cmap=cmocean.cm.speed,
                   vmin=0, vmax=ocean_clim, alpha=0.7)
fig.colorbar(im, ax=ax, label=r"$\| \mathbf{u}_o \|$  (m s$^{-1}$, mid-window)")

ax.plot(true_traj[:, 1],  true_traj[:, 0],  color="tab:blue",   lw=2,            label="truth")
ax.plot(init_traj[:, 1],  init_traj[:, 0],  color="tab:red",    lw=1.5, ls="--", label=r"initial ($\beta_w=0$)")
ax.plot(final_traj[:, 1], final_traj[:, 0], color="tab:orange", lw=2,   ls=":",  label="fitted")

ax.set_aspect("equal")
ax.set_xlabel("lon"); ax.set_ylabel("lat")
ax.legend(loc="upper right")
plt.show()


### 3.2 Stochastic: jointly learning $(\beta_w, C_S)$

The simulator is now stochastic. The truth is generated by the same SDE as the model,
with $(\beta_w^* = 3\%, \, C_S^* = 0.2)$. We observe **eight drifter trajectories**
launched from a small array of starting positions — each is one realisation of the SDE.

A scalar least-squares loss does not work for a stochastic simulator: a single
realisation is just one sample from the model distribution, and several different
$(\beta_w, C_S)$ may explain it equally well. We minimise instead the
**time-aggregated energy score** with separation distance as kernel, summed over
both observation time and observed drifter:

$$
\mathrm{ES}_t(F, y^*_t) = \frac{1}{M} \sum_{i=1}^M d(X^{(i)}_t, y^*_t) \;-\; \frac{1}{2M(M-1)} \sum_{i \neq j} d(X^{(i)}_t, X^{(j)}_t),
\qquad \mathcal{L} = \sum_{k=1}^K \sum_t \mathrm{ES}_t(F_k, y^{*,k}_t),
$$

where $d$ is the great-circle distance, $X^{(1)}, \dots, X^{(M)}$ is the model SDE
ensemble for drifter $k$, and $y^{*,k}$ is the corresponding observation. The energy
score is strictly proper: in expectation it is minimised when the simulator
distribution matches the truth.

We use a tiny ensemble ($M = 4$) per drifter and a short lead time (4 days) to keep
the optimisation tractable in a notebook.

In [ ]:
from molisanax import haversine

class TunableStoch(eqx.Module):
    beta_w: jax.Array
    c_s: jax.Array

    def __call__(self, t, y, args, z):
        dataset = args
        return smag_windage_term(t, y, (dataset, self.beta_w, self.c_s), z)

# Drifter array: eight starting positions on a small ring around y0
K = 8
ring_angles = 2 * jnp.pi * jnp.arange(K) / K
y0_drifters = jnp.stack([
    y0[0] + 0.04 * jnp.sin(ring_angles),
    y0[1] + 0.04 * jnp.cos(ring_angles),
], axis=-1)

# Truth observations: one SDE realisation per drifter, with (BETA_W_TRUE, CS_TRUE)
truth_term = TunableStoch(beta_w=jnp.array(BETA_W_TRUE), c_s=jnp.array(CS_TRUE))
truth_keys = jr.split(jr.key(123), K)
truth_drifters = jax.vmap(
    lambda y0_, k_: solve(truth_term, forcing, y0_, ts_sim, key=k_, n_noise=2)
)(y0_drifters, truth_keys)
print("truth drifter array shape:", truth_drifters.shape)  # (K, T, 2)

stoch_init = TunableStoch(beta_w=jnp.array(0.0), c_s=jnp.array(0.1))


In [ ]:
def energy_score(ensemble_traj, ref_traj):
    """Time-aggregated energy score with separation distance as kernel.

    ensemble_traj: (M, T, 2) lat/lon (deg).
    ref_traj:      (T, 2).
    Returns a scalar.
    """
    M = ensemble_traj.shape[0]
    sep_to_ref = separation_distance(ensemble_traj, ref_traj, ensemble=True)  # (M, T)
    term1 = sep_to_ref.mean(axis=0)  # (T,)

    def pair_sep(xi, xj):
        return jax.vmap(haversine)(xi, xj)
    pairs = jax.vmap(jax.vmap(pair_sep, in_axes=(None, 0)), in_axes=(0, None))(
        ensemble_traj, ensemble_traj
    )  # (M, M, T) -- diagonals are 0
    term2 = pairs.sum(axis=(0, 1)) / (2 * M * (M - 1))

    return jnp.sum(term1 - term2)


In [ ]:
M = 4  # tiny ensemble per drifter during optimisation

@eqx.filter_jit
def stoch_loss(term_module, args):
    obs_drifters, base_key = args
    keys_per_drifter = jr.split(base_key, K)
    def es_one(y0_, k_, y_obs):
        ens = solve(term_module, forcing, y0_, ts_sim,
                    key=k_, n_noise=2, n_samples=M)
        return energy_score(ens, y_obs)
    return jnp.sum(jax.vmap(es_one)(y0_drifters, keys_per_drifter, obs_drifters))

minimiser = optx.BFGS(rtol=1e-4, atol=1e-4)
sol_stoch = optx.minimise(
    stoch_loss, solver=minimiser, y0=stoch_init,
    args=(truth_drifters, jr.key(42)),
    max_steps=60, throw=False,
)
stoch_fit = sol_stoch.value
print(f"converged in {int(sol_stoch.stats['num_steps'])} steps  ->  "
      f"beta_w = {float(stoch_fit.beta_w) * 100:.3f}%, "
      f"C_S = {float(stoch_fit.c_s):.4f}")
print(f"truth                              ->  "
      f"beta_w = {BETA_W_TRUE * 100:.3f}%, C_S = {CS_TRUE:.4f}")


In [ ]:
# Draw a larger ensemble at one drifter with the fitted parameters
ens_fit = solve(stoch_fit, forcing, y0_drifters[0], ts_sim,
                key=jr.key(7), n_noise=2, n_samples=100)
ens_fit_np = np.asarray(ens_fit)
truth_np = np.asarray(truth_drifters)

fig, ax = plt.subplots(figsize=(5, 5), constrained_layout=True)
im = ax.pcolormesh(lon, lat, ocean_speed[len(ocean_speed) // 2],
                   cmap=cmocean.cm.speed, vmin=0, vmax=ocean_clim, alpha=0.6)
fig.colorbar(im, ax=ax, label=r"$\| \mathbf{u}_o \|$  (m s$^{-1}$, mid-window)")

# Fitted ensemble (for drifter 0): grey spaghetti
for i in range(ens_fit_np.shape[0]):
    ax.plot(ens_fit_np[i, :, 1], ens_fit_np[i, :, 0],
            color="black", alpha=0.10, lw=0.6)
# All eight observed drifters: blue
for k_ in range(K):
    ax.plot(truth_np[k_, :, 1], truth_np[k_, :, 0],
            color="tab:blue", lw=1.2, alpha=0.9,
            label="observed drifters" if k_ == 0 else None)
ax.scatter(y0_drifters[:, 1], y0_drifters[:, 0],
           color="white", edgecolor="black", zorder=3, label="starts")

ax.set_aspect("equal")
ax.set_xlabel("lon"); ax.set_ylabel("lat")
ax.legend(loc="upper right")
plt.show()


## Where to next

- See the [API reference](api.md) for the full surface of `solve`, `Dataset`, and the
  metric / interpolation helpers.
- Replace `Dataset.from_arrays` with `Dataset.from_xarray` to drive the simulator from
  real zarr or netCDF forcing fields.
- The `solve` integrator supports both reverse-mode (`jax.grad`, used by `BFGS`) and
  forward-mode (`jax.jvp`, used by `GaussNewton`) auto-differentiation.